## Raport: Segmentacja naczyń krwionośnych dna oka ##

**Autor:** Aleks Czarnecki 160190<br>
**Zadanie:** segmentacja naczyń krwionośnych siatkówki<br>
**Metody:** filtr Sato, Random Forest na cechach 5x5, CNN PyTorch<br>
**Język:** Python (3.14.3)

### 1. Konfiguracja notatnika


In [ ]:
import os
import sys
import tempfile
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import skimage
import tifffile
import torch
import torch.nn as nn
from PIL import Image
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from torch.utils.data import DataLoader, TensorDataset

BASE_DIR = Path.cwd()
DATASET_DIR = BASE_DIR / 'dataset'
RESULTS_DIR = BASE_DIR / 'wyniki'
COMPARISON_DIR = RESULTS_DIR / 'porownania'
MASKS_DIR = RESULTS_DIR / 'maski'

print('Wersje bibliotek:')
print('Python:', sys.version.split()[0])
print('numpy:', np.__version__)
print('matplotlib:', matplotlib.__version__)
print('Pillow:', Image.__version__)
print('scikit-image:', skimage.__version__)
print('tifffile:', tifffile.__version__)
print('scikit-learn: RandomForestClassifier dostępny')
print('torch:', torch.__version__)
print('')
print('Katalog notatnika:', BASE_DIR.resolve())
print('Katalog danych:', DATASET_DIR.resolve())

### 2. Dane wejściowe

Zbiór danych składa się z kolorowych fotografii dna oka, masek eksperckich naczyń oraz masek pola widzenia. Identyfikator obrazu ma postać `numer_wariant`, np. `01_h`, `14_g`, `15_dr`.

- `dataset/images` - obrazy wejściowe JPG,
- `dataset/manual1` - ręczne maski naczyń w formacie TIF,
- `dataset/mask` - maski obszaru analizowanego oka.

Metryki liczone są wyłącznie w obszarze maski pola widzenia, ponieważ tło poza siatkówką nie jest diagnostycznie istotne i sztucznie zawyżałoby accuracy.

In [ ]:
CONFIG = {
    'margin': 10,
    'patch_size': 5,
    'sato_sigmas': range(1, 5),
    'clahe_clip': 0.03,
    'random_state': 42,
    'max_samples_per_class_per_image': 4500,
    'prediction_chunk_size': 150000,
    'cnn_patch_size': 17,
    'cnn_epochs': 10,
    'cnn_batch_size': 768,
    'cnn_learning_rate': 0.0007,
    'cnn_weight_decay': 0.0001,
    'cnn_prediction_chunk_size': 50000,
    'cnn_validation_fraction': 0.20,
    'cnn_max_positive_samples_per_image': 3200,
    'cnn_negative_to_positive_ratio': 2.5,
    'cnn_threshold_min': 0.40,
    'cnn_threshold_max': 0.95,
}

BASIC_TEST_IDS = ['01_h', '02_h', '03_h', '04_h', '05_h']

TRAIN_IDS = [
    f'{idx:02d}_{wariant}'
    for wariant in ['h', 'g', 'dr']
    for idx in range(1, 6)
]

HOLDOUT_TEST_IDS = [
    f'{idx:02d}_{wariant}'
    for wariant in ['h', 'g', 'dr']
    for idx in [14, 15]
]

print('Obrazy do testu Sato:', BASIC_TEST_IDS)
print('Obrazy treningowe RF/CNN:', TRAIN_IDS)
print('Obrazy hold-out:', HOLDOUT_TEST_IDS)


### 3. Funkcje wczytywania i przygotowania danych

Obrazy kolorowe są normalizowane do zakresu `[0, 1]`. Maski binarne są progowane: jasne piksele oznaczają naczynia albo poprawny obszar pola widzenia. Margines jest usuwany, aby ograniczyć wpływ artefaktów brzegowych.

In [ ]:
def sciezka_obrazu(image_id):
    ''' Zwraca ścieżkę do obrazu o danym identyfikatorze '''
    for ext in ('.jpg', '.JPG', '.jpeg', '.JPEG'):
        path = DATASET_DIR / 'images' / f'{image_id}{ext}'
        if path.exists():
            return path
    raise FileNotFoundError(f'Brak obrazu dla identyfikatora: {image_id}')


def sciezka_maski_expert(image_id):
    ''' Zwraca ścieżkę do maski eksperckiej dla danego identyfikatora obrazu '''
    return DATASET_DIR / 'manual1' / f'{image_id}.tif'


def sciezka_maski_pola(image_id):
    ''' Zwraca ścieżkę do maski pola widzenia dla danego identyfikatora obrazu '''
    return DATASET_DIR / 'mask' / f'{image_id}_mask.tif'


def wczytaj_obraz_rgb(sciezka):
    ''' Wczytuje obraz RGB z podanej ścieżki i normalizuje wartości pikseli do zakresu [0, 1] '''
    with Image.open(sciezka) as image:
        return np.asarray(image.convert('RGB'), dtype=np.float32) / 255.0


def wczytaj_obraz(sciezka):
    ''' Wczytuje obraz w odcieniach szarości z podanej ścieżki i normalizuje wartości pikseli do zakresu [0, 1] '''
    with Image.open(sciezka) as image:
        return np.asarray(image.convert('L'), dtype=np.float32) / 255.0


def wczytaj_maske(sciezka):
    ''' Wczytuje maskę binarną z podanej ścieżki. Jeśli maska ma 3 kanały, używa tylko pierwszego. Zwraca maskę jako tablicę uint8 z wartościami 0 i 1. '''
    maska = tifffile.imread(sciezka)
    if maska.ndim == 3:
        maska = maska[:, :, 0]
    return (maska > 128).astype(np.uint8)


def przytnij_margines(obraz, margin):
    ''' Usuwa margines o szerokości `margin` pikseli z każdej krawędzi obrazu. Jeśli margin jest 0, zwraca oryginalny obraz. '''
    if margin == 0:
        return obraz
    return obraz[margin:-margin, margin:-margin]


def pokaz_przyklad(image_id='01_h'):
    ''' Pokazuje przykładowy obraz, maskę ekspercką i maskę pola widzenia dla danego identyfikatora obrazu. '''
    rgb = wczytaj_obraz_rgb(sciezka_obrazu(image_id))
    expert = wczytaj_maske(sciezka_maski_expert(image_id))
    maska_pola = wczytaj_maske(sciezka_maski_pola(image_id))

    _, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(rgb)
    axes[0].set_title(f'Obraz wejściowy {image_id}')
    axes[0].axis('off')
    axes[1].imshow(expert, cmap='gray')
    axes[1].set_title('Maska ekspercka')
    axes[1].axis('off')
    axes[2].imshow(maska_pola, cmap='gray')
    axes[2].set_title('Maska pola widzenia')
    axes[2].axis('off')
    plt.tight_layout()
    plt.show()


pokaz_przyklad('01_h')

### 4. Metoda klasyczna: CLAHE + filtr Sato + Otsu

Pierwszy wariant jest bez uczenia. Obraz jest wzmacniany przez CLAHE, następnie filtr Sato podbija struktury podobne do naczyń. Wynik jest normalizowany, progowany metodą Otsu i czyszczony operacjami morfologicznymi.

In [ ]:
def normalizuj(obraz):
    '''Normalizuje obraz do zakresu [0, 1].'''
    return (obraz - obraz.min()) / (obraz.max() - obraz.min() + 1e-8)

def zastosuj_clahe(obraz, clip_limit=0.03):
    '''Adaptive histogram equalization do wstępnego przetwarzania.'''
    return skimage.exposure.equalize_adapthist(obraz, clip_limit=clip_limit)

def zastosuj_sato(obraz, sigmas=range(1, 5)):
    '''Filtr Sato - detektor struktur tubularnych podobnych do naczyń.'''
    return skimage.filters.sato(obraz, sigmas=sigmas, black_ridges=True)

def binarnizuj_otsu(obraz_norm):
    '''Binaryzuje obraz automatycznym progiem Otsu.'''
    otsu_thresh = skimage.filters.threshold_otsu(obraz_norm)
    return (obraz_norm > otsu_thresh).astype(np.uint8)

def usun_artefakty(maska_bin, maska_pola=None):
    '''Post-processing: domknięcie morfologiczne i usunięcie drobnych obiektów.'''
    kernel = skimage.morphology.disk(2)
    maska = skimage.morphology.closing(maska_bin.astype(bool), kernel)
    maska = skimage.morphology.remove_small_objects(maska, max_size=19)
    if maska_pola is not None:
        maska &= maska_pola.astype(bool)
    return maska.astype(np.uint8)

def segmentuj_naczynia(obraz_raw, maska_pola=None, margin=10):
    '''Pipeline: crop -> CLAHE -> Sato -> normalizacja -> Otsu -> morfologia.'''
    obraz_crop = przytnij_margines(obraz_raw, margin)
    maska_pola_crop = przytnij_margines(maska_pola, margin) if maska_pola is not None else None

    obraz_clahe = zastosuj_clahe(obraz_crop, clip_limit=CONFIG['clahe_clip'])
    obraz_sato = zastosuj_sato(obraz_clahe, sigmas=CONFIG['sato_sigmas'])
    obraz_norm = normalizuj(obraz_sato)
    obraz_bin = binarnizuj_otsu(obraz_norm)
    obraz_bin = usun_artefakty(obraz_bin, maska_pola_crop)

    return {
        'preprocessed': obraz_clahe,
        'sato': obraz_norm,
        'segmentacja': obraz_bin,
    }


def segmentuj_naczynia_sato(obraz_raw, maska_pola=None, margin=10):
    '''Alias używany w raporcie dla klasycznego pipeline Sato.'''
    return segmentuj_naczynia(obraz_raw, maska_pola, margin=margin)


def pokaz_pipeline_sato(image_id='01_h'):
    '''Wyświetl pipeline segmentacji naczyń metodą Sato.'''
    obraz = wczytaj_obraz(sciezka_obrazu(image_id))
    maska_pola = wczytaj_maske(sciezka_maski_pola(image_id))
    wynik = segmentuj_naczynia_sato(obraz, maska_pola, margin=CONFIG['margin'])
    obraz_crop = przytnij_margines(obraz, CONFIG['margin'])

    _, axes = plt.subplots(1, 4, figsize=(20, 5))
    axes[0].imshow(obraz_crop, cmap='gray')
    axes[0].set_title('Obraz szary')
    axes[0].axis('off')
    axes[1].imshow(wynik['preprocessed'], cmap='gray')
    axes[1].set_title('CLAHE')
    axes[1].axis('off')
    axes[2].imshow(wynik['sato'], cmap='gray')
    axes[2].set_title('Odpowiedź filtra Sato')
    axes[2].axis('off')
    axes[3].imshow(wynik['segmentacja'], cmap='gray')
    axes[3].set_title('Segmentacja po Otsu')
    axes[3].axis('off')
    plt.tight_layout()
    plt.show()


pokaz_pipeline_sato('01_h')


### 5. Metryki jakości segmentacji

Klasy są silnie niezrównoważone: większość pikseli to tło, a naczynia zajmują relatywnie małą część obrazu. Dlatego samo accuracy nie wystarcza. Raport liczy również czułość, swoistość, precyzję, F1 oraz balanced accuracy.

In [ ]:
def policz_metryki(y_true, y_pred, valid_mask=None):
    ''' Oblicz metryki klasyfikacji binarnej, opcjonalnie tylko dla pikseli wskazanych przez valid_mask.'''
    y_true_flat = np.asarray(y_true).flatten()
    y_pred_flat = np.asarray(y_pred).flatten()

    if valid_mask is not None:
        valid_flat = np.asarray(valid_mask).astype(bool).flatten()
        y_true_flat = y_true_flat[valid_flat]
        y_pred_flat = y_pred_flat[valid_flat]

    cm = confusion_matrix(y_true_flat, y_pred_flat, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    sensitivity = recall_score(y_true_flat, y_pred_flat, zero_division=0)
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

    return {
        'accuracy': accuracy_score(y_true_flat, y_pred_flat),
        'precision': precision_score(y_true_flat, y_pred_flat, zero_division=0),
        'sensitivity': sensitivity,
        'specificity': specificity,
        'f1': f1_score(y_true_flat, y_pred_flat, zero_division=0),
        'balanced_accuracy_arithmetic': (sensitivity + specificity) / 2,
        'balanced_accuracy_geometric': np.sqrt(sensitivity * specificity),
        'tp': tp,
        'tn': tn,
        'fp': fp,
        'fn': fn,
        'confusion_matrix': cm,
    }


def wypisz_metryki(metryki, label):
    ''' Wypisz metryki klasyfikacji w czytelnej formie.'''
    tn, fp, fn, tp = metryki['tn'], metryki['fp'], metryki['fn'], metryki['tp']
    print(f'Metryki {label}:')
    print('  Macierz pomyłek [[TN, FP], [FN, TP]]:')
    print(f'  [[{tn}  {fp}] [{fn}  {tp}]]')
    print(f"  TP={metryki['tp']}, TN={metryki['tn']}, FP={metryki['fp']}, FN={metryki['fn']}\n")
    print(f"  Accuracy:          {metryki['accuracy']:.3f}")
    print(f"  Sensitivity:       {metryki['sensitivity']:.3f}")
    print(f"  Specificity:       {metryki['specificity']:.3f}")
    print(f"  Precision:         {metryki['precision']:.3f}")
    print(f"  F1-score:          {metryki['f1']:.3f}")
    print(f"  Balanced Acc (AM): {metryki['balanced_accuracy_arithmetic']:.3f}")
    print(f"  Balanced Acc (GM): {metryki['balanced_accuracy_geometric']:.3f}")


def podsumuj_wyniki(wyniki_wszystkie, naglowek=None):
    ''' Podsumuj wyniki z wielu obrazów, wypisując średnie i odchylenia standardowe metryk.'''
    if naglowek is not None:
        print('\n' + '=' * 70)
        print(naglowek)
        print('=' * 70)
    keys = ['accuracy', 'sensitivity', 'specificity', 'precision', 'f1', 'balanced_accuracy_arithmetic', 'balanced_accuracy_geometric']
    labels = ['Accuracy', 'Sensitivity', 'Specificity', 'Precision', 'F1-score', 'Balanced Acc (AM)', 'Balanced Acc (GM)']
    for key, label in zip(keys, labels):
        values = [m[key] for m in wyniki_wszystkie]
        print(f'{label:30} {np.mean(values):.3f} +/- {np.std(values):.3f}')

image_id_metryki = '01_h'
obraz_metryki = wczytaj_obraz(sciezka_obrazu(image_id_metryki))
maska_expert_metryki = wczytaj_maske(sciezka_maski_expert(image_id_metryki))
maska_pola_metryki = wczytaj_maske(sciezka_maski_pola(image_id_metryki))

wynik_sato_metryki = segmentuj_naczynia_sato(
    obraz_metryki,
    maska_pola_metryki,
    margin=CONFIG['margin'],
)

maska_expert_crop_metryki = przytnij_margines(maska_expert_metryki, CONFIG['margin'])
maska_pola_crop_metryki = przytnij_margines(maska_pola_metryki, CONFIG['margin'])
segmentacja_crop_metryki = wynik_sato_metryki['segmentacja']

metryki_przyklad = policz_metryki(
    maska_expert_crop_metryki,
    segmentacja_crop_metryki,
    maska_pola_crop_metryki,
)
wypisz_metryki(metryki_przyklad, f'Sato {image_id_metryki}')


### 6. Wizualizacja wyników

Dla każdej metody zapisywane jest porównanie: maska ekspercka oraz mapa błędów segmentacji względem tej maski. Na mapie błędów kolor zielony oznacza True Positive, czerwony False Positive, a niebieski False Negative.

In [ ]:
def naloz_segmentacje(rgb_crop, segmentacja):
    ''' Nałóż segmentację na obraz RGB, podświetlając wykryte naczynia na biało'''
    overlay = rgb_crop.copy()
    vessel = segmentacja.astype(bool)
    overlay[vessel] = 0.45 * overlay[vessel] + 0.55 * np.array([1.0, 1.0, 1.0])
    return np.clip(overlay, 0, 1)

def wyswietl_porownanie(rgb_crop, maska_expert, segmentacja, metryki, image_id, prefix, tytul):
    ''' Wyświetl na 3 kolumnach: maskę ekspercką, wynik segmentacji na RGB oraz mapę błędów.'''
    _, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].imshow(maska_expert, cmap='gray')
    axes[0].set_title('Maska ekspercka')
    axes[0].axis('off')
    
    axes[1].imshow(segmentacja, cmap='gray')
    axes[1].set_title(f'{tytul}')
    axes[1].axis('off')
    
    error_map = np.zeros((*maska_expert.shape, 3))
    tp_mask = (maska_expert == 1) & (segmentacja == 1)
    fp_mask = (maska_expert == 0) & (segmentacja == 1)
    fn_mask = (maska_expert == 1) & (segmentacja == 0)
    error_map[tp_mask] = [0, 1, 0]
    error_map[fp_mask] = [1, 0, 0]
    error_map[fn_mask] = [0.0, 0.2, 1.0]
    axes[2].imshow(error_map)
    axes[2].set_title(
        f'TP zielony | FP czerwony | FN niebieski\n'
        f'TP={metryki["tp"]} | FP={metryki["fp"]} | FN={metryki["fn"]}\n'
        f'F1={metryki["f1"]:.3f}'
    )
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()


image_id_wiz = '01_h'
rgb_wiz = wczytaj_obraz_rgb(sciezka_obrazu(image_id_wiz))
obraz_wiz = wczytaj_obraz(sciezka_obrazu(image_id_wiz))
maska_expert_wiz = wczytaj_maske(sciezka_maski_expert(image_id_wiz))
maska_pola_wiz = wczytaj_maske(sciezka_maski_pola(image_id_wiz))

wynik_sato_wiz = segmentuj_naczynia_sato(obraz_wiz, maska_pola_wiz, margin=CONFIG['margin'])
rgb_crop_wiz = przytnij_margines(rgb_wiz, CONFIG['margin'])
maska_expert_crop_wiz = przytnij_margines(maska_expert_wiz, CONFIG['margin'])
maska_pola_crop_wiz = przytnij_margines(maska_pola_wiz, CONFIG['margin'])
segmentacja_crop_wiz = wynik_sato_wiz['segmentacja']

metryki_wiz = policz_metryki(maska_expert_crop_wiz, segmentacja_crop_wiz, maska_pola_crop_wiz)

wyswietl_porownanie(
    rgb_crop_wiz,
    maska_expert_crop_wiz,
    segmentacja_crop_wiz,
    metryki_wiz,
    image_id_wiz,
    'porownanie_sato',
    'Segmentacja Sato',
)

### 7. Segmentacja filtrem Sato na zbiorze hold-out

Test obejmuje te same obrazy hold-out co Random Forest i CNN w punktach 8 oraz 9, czyli `14_*` i `15_*` dla wariantów `h`, `g`, `dr`. Dzięki temu porównanie metod odbywa się na identycznym zbiorze testowym. Sato jest wariantem szybkim i interpretowalnym, ale bez uczenia na danych.

In [ ]:
def uruchom_sato():
    '''Uruchamia klasyczny algorytm przetwarzania obrazu na 5 obrazach.'''
    wyniki_wszystkie = []
    print(f'Segmentacja naczyń filtrem Sato - {len(BASIC_TEST_IDS)} obrazów...')

    for image_id in BASIC_TEST_IDS:
        rgb = wczytaj_obraz_rgb(sciezka_obrazu(image_id))
        obraz = wczytaj_obraz(sciezka_obrazu(image_id))
        maska_expert = wczytaj_maske(sciezka_maski_expert(image_id))
        maska_pola = wczytaj_maske(sciezka_maski_pola(image_id))

        wyniki = segmentuj_naczynia(obraz, maska_pola, margin=CONFIG['margin'])
        rgb_crop = przytnij_margines(rgb, CONFIG['margin'])
        maska_expert_crop = przytnij_margines(maska_expert, CONFIG['margin'])
        maska_pola_crop = przytnij_margines(maska_pola, CONFIG['margin'])
        segmentacja_crop = wyniki['segmentacja']

        metryki = policz_metryki(maska_expert_crop, segmentacja_crop, maska_pola_crop)
        wyniki_wszystkie.append(metryki)
        wypisz_metryki(metryki, f'Sato {image_id}')
        wyswietl_porownanie(
            rgb_crop,
            maska_expert_crop,
            segmentacja_crop,
            metryki,
            image_id,
            'porownanie_sato',
            'Segmentacja Sato',
        )

    podsumuj_wyniki(wyniki_wszystkie, 'PODSUMOWANIE SATO - metryki średnie dla 5 obrazów')
    return wyniki_wszystkie


wyniki_sato = uruchom_sato()


### 8. Klasyfikator Random Forest na cechach z okien 5x5

Druga metoda uczy klasyfikator pikselowy. Dla każdego analizowanego piksela liczone są cechy z okna `5x5` na kanałach: RGB, zielony po CLAHE, odpowiedź Sato i Sobel. Do treningu pobierane są zbalansowane próbki naczyń i tła, a test odbywa się na niezależnym zbiorze hold-out (`14_*`, `15_*`).

In [ ]:
def przygotuj_kanaly_cech(rgb_crop):
    '''Tworzy kanały, z których będą liczone cechy okien 5x5.'''
    red = rgb_crop[:, :, 0]
    green = rgb_crop[:, :, 1]
    blue = rgb_crop[:, :, 2]
    green_clahe = zastosuj_clahe(green, clip_limit=CONFIG['clahe_clip'])
    sato = normalizuj(zastosuj_sato(green_clahe, sigmas=CONFIG['sato_sigmas']))
    sobel = normalizuj(skimage.filters.sobel(green_clahe))

    return {
        'red': red,
        'green': green,
        'blue': blue,
        'green_clahe': green_clahe,
        'sato': sato,
        'sobel': sobel,
    }

def wyznacz_cechy_z_okien(kanaly, coords, patch_size=5):
    '''
    Ekstrakcja cech dla wycinków patch_size x patch_size.
    Decyzja klasy dotyczy środkowego piksela wycinka.
    '''
    coords = np.asarray(coords, dtype=np.int64)
    pad = patch_size // 2
    rr = coords[:, 0]
    cc = coords[:, 1]
    cechy = []

    grid = np.arange(-pad, pad + 1, dtype=np.float32)
    xx, yy = np.meshgrid(grid, grid)

    for nazwa, kanal in kanaly.items():
        padded = np.pad(kanal, pad_width=pad, mode='reflect')
        windows = skimage.util.view_as_windows(padded, (patch_size, patch_size))
        patches = windows[rr, cc].astype(np.float32)

        mean = patches.mean(axis=(1, 2))
        centered = patches - mean[:, None, None]

        cechy.extend([
            patches[:, pad, pad],
            mean,
            patches.std(axis=(1, 2)),
            patches.var(axis=(1, 2)),
            patches.min(axis=(1, 2)),
            patches.max(axis=(1, 2)),
            (patches.max(axis=(1, 2)) - patches.min(axis=(1, 2))),
        ])

        if nazwa in ('green_clahe', 'sato'):
            cechy.extend([
                (centered * xx).mean(axis=(1, 2)),
                (centered * yy).mean(axis=(1, 2)),
                (patches * xx * xx).mean(axis=(1, 2)),
                (patches * yy * yy).mean(axis=(1, 2)),
                (patches * xx * yy).mean(axis=(1, 2)),
            ])

    return np.column_stack(cechy).astype(np.float32)

def losuj_wspolrzedne(mask_pos, mask_valid, limit, rng):
    '''Losuje współrzędne pikseli z maski pozytywnej, ograniczając do limitu.'''
    coords = np.column_stack(np.where(mask_pos & mask_valid))
    if len(coords) > limit:
        coords = coords[rng.choice(len(coords), size=limit, replace=False)]
    return coords

def zbuduj_zbior_uczacy(image_ids):
    '''Buduje zbalansowany zbiór uczący z losowo wybranych wycinków 5x5.'''
    rng = np.random.default_rng(CONFIG['random_state'])
    x_parts = []
    y_parts = []

    for image_id in image_ids:
        rgb = przytnij_margines(wczytaj_obraz_rgb(sciezka_obrazu(image_id)), CONFIG['margin'])
        expert = przytnij_margines(wczytaj_maske(sciezka_maski_expert(image_id)), CONFIG['margin']).astype(bool)
        maska_pola = przytnij_margines(wczytaj_maske(sciezka_maski_pola(image_id)), CONFIG['margin']).astype(bool)

        kanaly = przygotuj_kanaly_cech(rgb)
        limit = CONFIG['max_samples_per_class_per_image']
        pos_coords = losuj_wspolrzedne(expert, maska_pola, limit, rng)
        neg_coords = losuj_wspolrzedne(~expert, maska_pola, len(pos_coords), rng)

        coords = np.vstack([pos_coords, neg_coords])
        y = np.concatenate([
            np.ones(len(pos_coords), dtype=np.uint8),
            np.zeros(len(neg_coords), dtype=np.uint8),
        ])

        order = rng.permutation(len(coords))
        coords = coords[order]
        y = y[order]

        x_parts.append(wyznacz_cechy_z_okien(kanaly, coords, CONFIG['patch_size']))
        y_parts.append(y)
        print(f'  {image_id}: próbki uczące={len(y)} (naczynia={int(y.sum())}, tło={int((y == 0).sum())})')

    return np.vstack(x_parts), np.concatenate(y_parts)

def trenuj_klasyfikator(image_ids):
    '''Trenuje prosty klasyfikator scikit-learn na cechach z okien 5x5.'''
    print('\nTrening klasyfikatora RandomForest na cechach 5x5...')
    x_train, y_train = zbuduj_zbior_uczacy(image_ids)

    klasyfikator = RandomForestClassifier(
        n_estimators=80,
        max_depth=18,
        min_samples_leaf=3,
        class_weight='balanced_subsample',
        n_jobs=-1,
        random_state=CONFIG['random_state'],
    )
    klasyfikator.fit(x_train, y_train)
    print(f'  Razem próbek: {len(y_train)}, liczba cech: {x_train.shape[1]}')
    return klasyfikator

def predykcja_klasyfikatora(klasyfikator, rgb_raw, maska_pola_raw, margin=10):
    '''Predykcja maski naczyń dla całego obrazu z użyciem klasyfikatora.'''
    rgb = przytnij_margines(rgb_raw, margin)
    maska_pola = przytnij_margines(maska_pola_raw, margin).astype(bool)
    kanaly = przygotuj_kanaly_cech(rgb)

    coords = np.column_stack(np.where(maska_pola))
    segmentacja = np.zeros(maska_pola.shape, dtype=np.uint8)

    chunk = CONFIG['prediction_chunk_size']
    for start in range(0, len(coords), chunk):
        coords_chunk = coords[start:start + chunk]
        x_chunk = wyznacz_cechy_z_okien(kanaly, coords_chunk, CONFIG['patch_size'])
        segmentacja[coords_chunk[:, 0], coords_chunk[:, 1]] = klasyfikator.predict(x_chunk)

    return usun_artefakty(segmentacja, maska_pola)


In [ ]:
def uruchom_random_forest():
    '''Trenuje klasyfikator na cechach 5x5 i testuje go na niezależnym hold-out.'''
    klasyfikator = trenuj_klasyfikator(TRAIN_IDS)
    wyniki_wszystkie = []

    print('')
    print(f'Predykcja klasyfikatora na hold-out - {len(HOLDOUT_TEST_IDS)} obrazów...')
    for image_id in HOLDOUT_TEST_IDS:
        rgb = wczytaj_obraz_rgb(sciezka_obrazu(image_id))
        maska_expert = wczytaj_maske(sciezka_maski_expert(image_id))
        maska_pola = wczytaj_maske(sciezka_maski_pola(image_id))

        segmentacja = predykcja_klasyfikatora(klasyfikator, rgb, maska_pola, margin=CONFIG['margin'])
        rgb_crop = przytnij_margines(rgb, CONFIG['margin'])
        maska_expert_crop = przytnij_margines(maska_expert, CONFIG['margin'])
        maska_pola_crop = przytnij_margines(maska_pola, CONFIG['margin'])

        metryki = policz_metryki(maska_expert_crop, segmentacja, maska_pola_crop)
        wyniki_wszystkie.append(metryki)
        wypisz_metryki(metryki, f'RandomForest 5x5 {image_id}')
        wyswietl_porownanie(
            rgb_crop,
            maska_expert_crop,
            segmentacja,
            metryki,
            image_id,
            'porownanie_rf',
            'RandomForest, cechy 5x5',
        )

    podsumuj_wyniki(
        wyniki_wszystkie,
        'PODSUMOWANIE RANDOM FOREST 5x5 - niezależny zbiór hold-out',
    )
    return klasyfikator, wyniki_wszystkie


klasyfikator_rf, wyniki_rf = uruchom_random_forest()


### 9. Głęboka sieć CNN na patchach 17x17

Trzecia metoda korzysta z działającej wersji ze skryptu `dno_oka.py`: mała sieć CNN klasyfikuje środkowy piksel patcha `17x17` z kanałów `green_clahe`, `sato`, `sobel`. Model używa `AdaptiveAvgPool2d`, dzięki czemu klasyfikator jest prostszy i mniej zależny od ręcznie wyliczonego rozmiaru map cech. Próg decyzyjny jest dobierany na walidacji przez maksymalizację F1.


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset


def wyznacz_patche_cnn(kanaly, coords, patch_size=15):
    '''Tworzy tensory patchy dla sieci CNN z kanałów green_clahe, sato i sobel.'''
    coords = np.asarray(coords, dtype=np.int64)
    pad = patch_size // 2
    rr = coords[:, 0]
    cc = coords[:, 1]
    patches_all = []

    for nazwa in ('green_clahe', 'sato', 'sobel'):
        kanal = kanaly[nazwa]
        padded = np.pad(kanal, pad_width=pad, mode='reflect')
        windows = skimage.util.view_as_windows(padded, (patch_size, patch_size))
        patches_all.append(windows[rr, cc].astype(np.float32))

    return np.stack(patches_all, axis=1)

def zbuduj_zbior_uczacy_cnn(image_ids):
    '''Buduje zbiór uczący patchy dla CNN z większą liczbą przykładów tła.'''
    rng = np.random.default_rng(CONFIG['random_state'])
    x_parts = []
    y_parts = []

    for image_id in image_ids:
        rgb = przytnij_margines(wczytaj_obraz_rgb(sciezka_obrazu(image_id)), CONFIG['margin'])
        expert = przytnij_margines(wczytaj_maske(sciezka_maski_expert(image_id)), CONFIG['margin']).astype(bool)
        maska_pola = przytnij_margines(wczytaj_maske(sciezka_maski_pola(image_id)), CONFIG['margin']).astype(bool)

        kanaly = przygotuj_kanaly_cech(rgb)
        pos_limit = CONFIG['cnn_max_positive_samples_per_image']
        pos_coords = losuj_wspolrzedne(expert, maska_pola, pos_limit, rng)
        neg_limit = int(len(pos_coords) * CONFIG['cnn_negative_to_positive_ratio'])
        neg_coords = losuj_wspolrzedne(~expert, maska_pola, neg_limit, rng)

        coords = np.vstack([pos_coords, neg_coords])
        y = np.concatenate([
            np.ones(len(pos_coords), dtype=np.int64),
            np.zeros(len(neg_coords), dtype=np.int64),
        ])

        order = rng.permutation(len(coords))
        coords = coords[order]
        y = y[order]

        x_parts.append(wyznacz_patche_cnn(kanaly, coords, CONFIG['cnn_patch_size']))
        y_parts.append(y)
        print(f'  {image_id}: próbki CNN={len(y)} (naczynia={int(y.sum())}, tło={int((y == 0).sum())})')

    return np.vstack(x_parts), np.concatenate(y_parts)

class MalaSiecCNN(nn.Module):
    '''Prosta głęboka sieć CNN klasyfikująca środkowy piksel patcha.'''

    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 2),
        )

    def forward(self, x):
        '''Przekazuje dane przez sieć, zwracając logity dla dwóch klas.'''
        return self.model(x)

def wybierz_urzadzenie_torch():
    '''Wybiera MPS na Apple Silicon, jeśli jest dostępne, w innym razie CPU.'''
    if torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')

def policz_prawdopodobienstwa_cnn(model, loader, device):
    '''Zwraca prawdopodobieństwa klasy naczynia i etykiety dla podanego loadera.'''
    probs_parts = []
    y_parts = []

    model.eval()
    with torch.no_grad():
        for x_batch, y_batch in loader:
            logits = model(x_batch.to(device))
            probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
            probs_parts.append(probs)
            y_parts.append(y_batch.numpy())

    return np.concatenate(probs_parts), np.concatenate(y_parts)

def dobierz_prog_cnn(model, loader, device):
    '''Dobiera próg decyzji CNN na walidacji, maksymalizując F1.'''
    probs, y_true = policz_prawdopodobienstwa_cnn(model, loader, device)
    najlepszy_prog = 0.5
    najlepszy_f1 = -1.0

    for prog in np.linspace(CONFIG['cnn_threshold_min'], CONFIG['cnn_threshold_max'], 56):
        y_pred = (probs >= prog).astype(np.uint8)
        wynik_f1 = f1_score(y_true, y_pred, zero_division=0)
        if wynik_f1 > najlepszy_f1:
            najlepszy_f1 = wynik_f1
            najlepszy_prog = float(prog)

    return najlepszy_prog, najlepszy_f1

def trenuj_cnn(image_ids):
    '''Trenuje głęboką sieć CNN PyTorch na patchach obrazu.'''
    print('\nTrening głębokiej sieci CNN PyTorch na większym zbiorze patchy obrazu...')
    torch.manual_seed(CONFIG['random_state'])
    np.random.seed(CONFIG['random_state'])

    x_train, y_train = zbuduj_zbior_uczacy_cnn(image_ids)
    rng = np.random.default_rng(CONFIG['random_state'])
    order = rng.permutation(len(y_train))
    val_size = int(len(y_train) * CONFIG['cnn_validation_fraction'])
    val_idx = order[:val_size]
    train_idx = order[val_size:]

    train_dataset = TensorDataset(
        torch.from_numpy(x_train[train_idx]).float(),
        torch.from_numpy(y_train[train_idx]).long(),
    )
    val_dataset = TensorDataset(
        torch.from_numpy(x_train[val_idx]).float(),
        torch.from_numpy(y_train[val_idx]).long(),
    )
    train_loader = DataLoader(
        train_dataset,
        batch_size=CONFIG['cnn_batch_size'],
        shuffle=True,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=CONFIG['cnn_batch_size'],
        shuffle=False,
    )

    device = wybierz_urzadzenie_torch()
    model = MalaSiecCNN().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=CONFIG['cnn_learning_rate'],
        weight_decay=CONFIG['cnn_weight_decay'],
    )

    print(
        f'  Razem próbek: {len(y_train)}, trening={len(train_dataset)}, '
        f'walidacja={len(val_dataset)}, kształt wejścia: {x_train.shape[1:]}, urządzenie: {device}, lr={CONFIG["cnn_learning_rate"]}'
    )
    for epoch in range(CONFIG['cnn_epochs']):
        total_loss = 0.0
        correct = 0
        total = 0

        model.train()
        for x_batch, y_batch in train_loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()
            logits = model(x_batch)
            loss = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * len(y_batch)
            predictions = logits.argmax(dim=1)
            correct += (predictions == y_batch).sum().item()
            total += len(y_batch)

        val_probs, val_true = policz_prawdopodobienstwa_cnn(model, val_loader, device)
        val_pred = (val_probs >= 0.5).astype(np.uint8)
        val_f1 = f1_score(val_true, val_pred, zero_division=0)
        print(
            f'  Epoka {epoch + 1}/{CONFIG["cnn_epochs"]}: '
            f'loss={total_loss / total:.4f}, accuracy={correct / total:.3f}, val_f1@0.50={val_f1:.3f}'
        )

    prog, val_f1 = dobierz_prog_cnn(model, val_loader, device)
    model.prog_decyzji = prog
    print(f'  Wybrany próg decyzji CNN: {prog:.2f} (F1 walidacyjne={val_f1:.3f})')
    return model

def predykcja_cnn(model, rgb_raw, maska_pola_raw, margin=10):
    '''Predykcja maski naczyń dla całego obrazu z użyciem głębokiej sieci CNN.'''
    rgb = przytnij_margines(rgb_raw, margin)
    maska_pola = przytnij_margines(maska_pola_raw, margin).astype(bool)
    kanaly = przygotuj_kanaly_cech(rgb)

    coords = np.column_stack(np.where(maska_pola))
    segmentacja = np.zeros(maska_pola.shape, dtype=np.uint8)
    device = next(model.parameters()).device
    chunk = CONFIG['cnn_prediction_chunk_size']

    model.eval()
    with torch.no_grad():
        for start in range(0, len(coords), chunk):
            coords_chunk = coords[start:start + chunk]
            x_chunk = wyznacz_patche_cnn(kanaly, coords_chunk, CONFIG['cnn_patch_size'])
            x_tensor = torch.from_numpy(x_chunk).float().to(device)
            probs = torch.softmax(model(x_tensor), dim=1)[:, 1].cpu().numpy()
            predictions = (probs >= model.prog_decyzji).astype(np.uint8)
            segmentacja[coords_chunk[:, 0], coords_chunk[:, 1]] = predictions

    return usun_artefakty(segmentacja, maska_pola)


In [ ]:
def uruchom_cnn():
    '''Trenuje głęboką sieć CNN i testuje ją na niezależnym hold-out.'''
    model = trenuj_cnn(TRAIN_IDS)
    wyniki_wszystkie = []

    print('')
    print(f'Predykcja CNN na hold-out - {len(HOLDOUT_TEST_IDS)} obrazów...')
    for image_id in HOLDOUT_TEST_IDS:
        rgb = wczytaj_obraz_rgb(sciezka_obrazu(image_id))
        maska_expert = wczytaj_maske(sciezka_maski_expert(image_id))
        maska_pola = wczytaj_maske(sciezka_maski_pola(image_id))

        segmentacja = predykcja_cnn(model, rgb, maska_pola, margin=CONFIG['margin'])
        rgb_crop = przytnij_margines(rgb, CONFIG['margin'])
        maska_expert_crop = przytnij_margines(maska_expert, CONFIG['margin'])
        maska_pola_crop = przytnij_margines(maska_pola, CONFIG['margin'])

        metryki = policz_metryki(maska_expert_crop, segmentacja, maska_pola_crop)
        wyniki_wszystkie.append(metryki)
        wypisz_metryki(metryki, f'CNN PyTorch {CONFIG["cnn_patch_size"]}x{CONFIG["cnn_patch_size"]} {image_id}')
        wyswietl_porownanie(
            rgb_crop,
            maska_expert_crop,
            segmentacja,
            metryki,
            image_id,
            'porownanie_cnn',
            f'CNN PyTorch, patche {CONFIG["cnn_patch_size"]}x{CONFIG["cnn_patch_size"]}',
        )

    podsumuj_wyniki(
        wyniki_wszystkie,
        f'PODSUMOWANIE CNN PYTORCH {CONFIG["cnn_patch_size"]}x{CONFIG["cnn_patch_size"]} - niezależny zbiór hold-out',
    )
    return model, wyniki_wszystkie


model_cnn, wyniki_cnn = uruchom_cnn()


### 10. Porównanie metod

Sato jest metodą szybką i prostą, ale działa głównie na kontraście i geometrii lokalnej. Random Forest wykorzystuje cechy z wielu kanałów i potrafi lepiej odróżniać naczynia od struktur tła. CNN uczy reprezentację bez ręcznego projektowania wszystkich statystyk, ale wymaga więcej czasu obliczeniowego i jest bardziej wrażliwa na dobór zbioru treningowego oraz progu decyzyjnego.

In [ ]:
def srednia_metryka(wyniki, key):
    ''' Oblicz średnią wartość metryki o nazwie `key` z listy wyników. '''
    return float(np.mean([m[key] for m in wyniki]))


porownanie = {
    'Sato': wyniki_sato,
    'Random Forest 5x5': wyniki_rf,
    'CNN 15x15': wyniki_cnn,
}

metryki_do_wykresu = ['sensitivity', 'specificity', 'precision', 'f1', 'balanced_accuracy_arithmetic']
etykiety = ['Sensitivity', 'Specificity', 'Precision', 'F1', 'Balanced Acc']
x = np.arange(len(metryki_do_wykresu))
width = 0.25

plt.figure(figsize=(12, 5))
for i, (nazwa, wyniki) in enumerate(porownanie.items()):
    wartosci = [srednia_metryka(wyniki, key) for key in metryki_do_wykresu]
    plt.bar(x + (i - 1) * width, wartosci, width=width, label=nazwa)

plt.xticks(x, etykiety)
plt.ylim(0, 1)
plt.ylabel('Średnia wartość metryki')
plt.title('Porównanie średnich metryk segmentacji')
plt.grid(axis='y', alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

print('Tabela porównawcza średnich metryk:')
for nazwa, wyniki in porownanie.items():
    print(f'\n{nazwa}')
    for key, label in zip(metryki_do_wykresu, etykiety):
        print(f'  {label:14} {srednia_metryka(wyniki, key):.3f}')

### 11. Wnioski

- Filtr Sato jest dobrym punktem odniesienia: jest szybki, interpretowalny i nie wymaga treningu, ale może pomijać cienkie naczynia oraz reagować na inne ciemne struktury.
- Random Forest poprawia separację klas dzięki cechom lokalnym z okien `5x5` oraz kanałom pochodnym (`CLAHE`, `Sato`, `Sobel`).
- CNN wykorzystuje patch `17x17` i prostą sieć z AdaptiveAvgPool2d i uczy cechy automatycznie, ale kosztuje więcej obliczeniowo i wymaga kontroli walidacyjnej progu.
- Najbardziej miarodajne porównanie metod powinno opierać się na F1, czułości, swoistości i balanced accuracy, liczonych wyłącznie wewnątrz maski pola widzenia.
